# 10D correlated non-product RBM — HCN pilot
从 Harrison notebook 扩展：内部和边界均有全纯交互分支。默认先运行 smoke；通过后把 RUN_MODE 改为 train，重启内核并 Run All。依赖：torch、numpy、matplotlib、mpmath。

固定模型：R=I，mu=-1，Sigma[i,j]=2 rho^|i-j|/sqrt(ij)，rho=0.3。评估 t=1,...,8。默认25k更新、64M基础输入。配置是待验证的pilot，不保证训练精度。

恢复训练：RUN_MODE='resume'，RUN_DIR 指向原输出目录。评估：RUN_MODE='evaluate'，同样设置 RUN_DIR。只对固定25k终点评估，不按参考概率挑选模型。

训练只使用RBM参数及结构损失。截图中的模拟参考仅供训练后pilot比较，不是已经核验的真值。
仓库公开的是固定 seed 12345 的 exploratory pilot。默认 `RUN_MODE="smoke"`。从仓库根目录启动 Jupyter；要复算已保存终点，将 `RUN_MODE="evaluate"` 且 `RUN_DIR="results/d10_correlated_nonproduct/seed12345"`，然后 Restart Kernel and Run All。训练则设为 `train`，新输出保存在当前工作目录 `outputs/`；评估参考是未经认证的 pilot，最终论文误差在 `paper/data/` 做 clipping。


In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import csv
import hashlib
import json
import math
import platform
import random
import sys
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import mpmath as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

r_dtype = torch.float64
c_dtype = torch.complex128
torch.set_default_dtype(r_dtype)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.use_deterministic_algorithms(True, warn_only=True)


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2), encoding="utf-8")


def write_csv(path, rows, fieldnames):
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def safe_torch_load(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


print({"python": sys.version.split()[0], "torch": torch.__version__, "device": str(device)})

In [ ]:
RUN_MODE = "smoke"
RUN_DIR = None
VARIANT = "interaction"
SEED = 12345
D = 10
RHO = 0.3
TRAIN_STEPS = 25_000
BATCH_BAR, BATCH_BOUNDARY, BATCH_MONO = 2048, 256, 256
LR_INITIAL, LR_FINAL, LR_SWITCH = 5e-5, 1e-5, 20_000
SAVE_EVERY, LOG_EVERY = 1000, 100
POLE_TOL = 1e-12
MODEL_CONFIG = dict(d=D, coord_embed_dim=16, boundary_embed_dim=16,
    interior_hidden_width=95, boundary_hidden_width=128,
    global_hidden_width=109, pair_chunk_size=4096)
# Conservative negative-real extent: sum_i a_i/i < 1 is a sufficient
# joint exponential-moment condition by generalized Holder, since Zi~Exp(i).
# 0.2 * sum(1/i) < 1, unlike the inherited uniform bound 0.5.
RE_MIN, RE_MAX, IM_MAX = -0.2, 5.0, 5.0
assert 0.2 * sum(1/i for i in range(1,D+1)) < 1
assert RUN_MODE in {'smoke','train','resume','evaluate'}
assert VARIANT in {'interaction','coordinate_only'}
CONFIG = dict(schema=1, variant=VARIANT, seed=SEED, d=D, rho=RHO,
    steps=TRAIN_STEPS, batch=[BATCH_BAR,BATCH_BOUNDARY,BATCH_MONO],
    lr=[LR_INITIAL,LR_FINAL,LR_SWITCH], model=MODEL_CONFIG,
    domain=[RE_MIN,RE_MAX,IM_MAX], loss_weights=[1.,1.,1.,1.], grad_clip=1.)
print('Device:', device, 'base inputs:', TRAIN_STEPS*(BATCH_BAR+BATCH_BOUNDARY+BATCH_MONO))


In [ ]:
@dataclass
class RBMParams:
    mu: torch.Tensor
    Sigma: torch.Tensor
    R: torch.Tensor
    alpha: torch.Tensor | None = None


def build_params():
    idx=torch.arange(1,D+1,dtype=r_dtype,device=device)
    sigma=2*RHO**torch.abs(idx[:,None]-idx[None,:])/torch.sqrt(idx[:,None]*idx[None,:])
    torch.linalg.cholesky(sigma)
    return RBMParams(-torch.ones(D,device=device),sigma,torch.eye(D,device=device))


def compute_boundary_mass(params):
    mass = -torch.linalg.solve(params.R, params.mu)
    if not torch.isfinite(mass).all() or torch.any(mass <= 0):
        raise ValueError("Boundary masses must be finite and positive.")
    return mass


def compute_gammas(theta, params):
    theta = theta.to(dtype=c_dtype, device=device)
    mu = params.mu.to(c_dtype)
    Sigma = params.Sigma.to(c_dtype)
    R = params.R.to(c_dtype)
    gamma0 = -0.5 * torch.sum((theta @ Sigma) * theta, dim=1, keepdim=True) + theta @ mu[:, None]
    gamma_all = -(theta @ R)
    return gamma0, gamma_all


def rbm_payload(params):
    return {
        "mu": params.mu.cpu().tolist(),
        "Sigma": params.Sigma.cpu().tolist(),
        "R": params.R.cpu().tolist(),
        "alpha": None,
    }


PARAMS = build_params()
assert torch.allclose(compute_boundary_mass(PARAMS), torch.ones(D, device=device))
print('RBM: d=10, rho=0.3, R=I; covariance Cholesky passed')


In [ ]:
class CauchyComplex(nn.Module):
    def __init__(self, num_features, init_lambda=0.1, init_d=0.5, d_min=1e-3):
        super().__init__()
        self.num_features = int(num_features)
        self.d_min = float(d_min)
        self.lambda1 = nn.Parameter(torch.ones(num_features, dtype=c_dtype) * complex(init_lambda, 0.0))
        self.lambda2 = nn.Parameter(torch.ones(num_features, dtype=c_dtype) * complex(init_lambda, 0.0))
        init_value = max(init_d - d_min, 1e-6)
        init_rho = torch.log(torch.expm1(torch.tensor(init_value, dtype=r_dtype)))
        self.rho = nn.Parameter(torch.ones(num_features, dtype=r_dtype) * init_rho)
        self.last_min_abs_denominator = None
        self.running_min_abs_denominator = None

    def positive_d(self):
        return F.softplus(self.rho) + self.d_min

    def reset_denominator_tracking(self):
        self.last_min_abs_denominator = None
        self.running_min_abs_denominator = None

    def _update_denominator_tracking(self, denominator):
        current = torch.min(torch.abs(denominator)).detach()
        self.last_min_abs_denominator = current
        self.running_min_abs_denominator = current if self.running_min_abs_denominator is None else torch.minimum(self.running_min_abs_denominator, current).detach()

    def _denominator(self, z):
        d_complex = self.positive_d().to(device=z.device, dtype=z.dtype)
        return z ** 2 + d_complex ** 2, d_complex

    def forward(self, z):
        denominator, _ = self._denominator(z)
        self._update_denominator_tracking(denominator)
        output = (self.lambda1 * z + self.lambda2) / denominator
        if not torch.isfinite(output.real).all() or not torch.isfinite(output.imag).all():
            raise FloatingPointError("Non-finite Cauchy activation output.")
        return output

    def derivative(self, z):
        denominator, d_complex = self._denominator(z)
        self._update_denominator_tracking(denominator)
        output = (self.lambda1 * (d_complex ** 2 - z ** 2) - 2.0 * self.lambda2 * z) / (denominator ** 2)
        if not torch.isfinite(output.real).all() or not torch.isfinite(output.imag).all():
            raise FloatingPointError("Non-finite Cauchy derivative output.")
        return output


def reset_all_cauchy_denominator_tracking(model):
    for module in model.modules():
        if isinstance(module, CauchyComplex):
            module.reset_denominator_tracking()


def minimum_tracked_cauchy_denominator(model):
    values = [module.running_min_abs_denominator for module in model.modules() if isinstance(module, CauchyComplex) and module.running_min_abs_denominator is not None]
    if not values:
        return math.inf
    return float(torch.min(torch.stack(values)).detach().cpu())


class HolomorphicCauchyScalarNet(nn.Module):
    """Complex Linear -> Cauchy -> Complex Linear scalar map."""
    def __init__(self, input_dim, hidden_width):
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_width = int(hidden_width)
        self.linear_0 = nn.Linear(input_dim, hidden_width, dtype=c_dtype)
        self.cauchy_0 = CauchyComplex(hidden_width)
        self.linear_1 = nn.Linear(hidden_width, 1, dtype=c_dtype)

    def forward(self, x):
        if x.ndim != 2 or x.shape[1] != self.input_dim:
            raise ValueError(f"x must have shape [N,{self.input_dim}], got {tuple(x.shape)}")
        x = x.to(dtype=c_dtype, device=device)
        return self.linear_1(self.cauchy_0(self.linear_0(x)))

    def forward_with_input_jacobian(self, x):
        x = x.to(dtype=c_dtype, device=device)
        z = self.linear_0(x)
        activation = self.cauchy_0(z)
        output = self.linear_1(activation)
        derivative = self.cauchy_0.derivative(z)
        jacobian = torch.einsum("h,nh,hj->nj", self.linear_1.weight[0], derivative, self.linear_0.weight)
        return output, jacobian

    def forward_with_first_input_derivative(self, x):
        output, jacobian = self.forward_with_input_jacobian(x)
        return output, jacobian[:, 0:1]


def run_scalar_net_chunked(net, x, pair_chunk_size=32768, derivative_mode=None):
    outputs, derivatives = [], []
    for start in range(0, x.shape[0], pair_chunk_size):
        chunk = x[start:start + pair_chunk_size]
        if derivative_mode == "first":
            output, derivative = net.forward_with_first_input_derivative(chunk)
            derivatives.append(derivative)
        elif derivative_mode == "all":
            output, derivative = net.forward_with_input_jacobian(chunk)
            derivatives.append(derivative)
        else:
            output = net(chunk)
        outputs.append(output)
    output = torch.cat(outputs, dim=0)
    return (output, torch.cat(derivatives, dim=0)) if derivative_mode is not None else output


class StructuredNonproductGlobalCauchyMGF(nn.Module):
    """Coordinate branch + full-input global branch with structural origin anchor."""
    def __init__(self, d, coord_embed_dim, boundary_embed_dim, interior_hidden_width, boundary_hidden_width, global_hidden_width, pair_chunk_size, boundary_mass):
        super().__init__()
        self.d = int(d)
        self.coord_embed_dim = int(coord_embed_dim)
        self.boundary_embed_dim = int(boundary_embed_dim)
        self.pair_chunk_size = int(pair_chunk_size)
        boundary_mass = boundary_mass.detach().clone().to(dtype=r_dtype, device=device)
        if boundary_mass.shape != (self.d,) or not torch.isfinite(boundary_mass).all() or torch.any(boundary_mass <= 0):
            raise ValueError("boundary_mass must be finite, positive, and have shape [d].")
        self.register_buffer("boundary_mass", boundary_mass, persistent=True)
        self.register_buffer("boundary_log_mass", torch.log(boundary_mass).to(c_dtype), persistent=True)
        scale = 0.01
        self.coord_embedding = nn.Parameter(scale * torch.complex(torch.randn(self.d, self.coord_embed_dim, dtype=r_dtype), torch.randn(self.d, self.coord_embed_dim, dtype=r_dtype)))
        self.boundary_embedding = nn.Parameter(scale * torch.complex(torch.randn(self.d, self.boundary_embed_dim, dtype=r_dtype), torch.randn(self.d, self.boundary_embed_dim, dtype=r_dtype)))
        self.interior_net = HolomorphicCauchyScalarNet(1 + self.coord_embed_dim, interior_hidden_width)
        self.boundary_net = HolomorphicCauchyScalarNet(1 + self.coord_embed_dim + self.boundary_embed_dim, boundary_hidden_width)
        self.global_net = HolomorphicCauchyScalarNet(self.d, global_hidden_width)

    def _interior_inputs(self, theta):
        batch, d = theta.shape
        embedding = self.coord_embedding.view(1, d, self.coord_embed_dim).expand(batch, d, self.coord_embed_dim)
        return torch.cat([theta.unsqueeze(-1), embedding], dim=2).reshape(batch * d, 1 + self.coord_embed_dim)

    def _interior_f0_and_jac(self, theta, need_jacobian):
        batch, d = theta.shape
        inputs = self._interior_inputs(theta)
        if need_jacobian:
            coordinate_values, coordinate_derivatives = run_scalar_net_chunked(self.interior_net, inputs, self.pair_chunk_size, derivative_mode="first")
            coordinate_derivatives = coordinate_derivatives.view(batch, d)
            global_values, global_jacobian = run_scalar_net_chunked(self.global_net, theta, self.pair_chunk_size, derivative_mode="all")
        else:
            coordinate_values = run_scalar_net_chunked(self.interior_net, inputs, self.pair_chunk_size)
            global_values = run_scalar_net_chunked(self.global_net, theta, self.pair_chunk_size)
            coordinate_derivatives = global_jacobian = None
        coordinate_values = coordinate_values.view(batch, d)
        zero_coordinate_inputs = torch.cat([torch.zeros(d, 1, dtype=c_dtype, device=theta.device), self.coord_embedding], dim=1)
        coordinate_zero = run_scalar_net_chunked(self.interior_net, zero_coordinate_inputs, self.pair_chunk_size).view(1, d)
        global_zero = run_scalar_net_chunked(self.global_net, torch.zeros(1, d, dtype=c_dtype, device=theta.device), self.pair_chunk_size)
        f_coordinate = torch.sum(coordinate_values - coordinate_zero, dim=1, keepdim=True)
        f_global = global_values - global_zero
        return f_coordinate + f_global, None if not need_jacobian else coordinate_derivatives + global_jacobian

    def _boundary_inputs_for_k(self, theta, k):
        batch, d = theta.shape
        mask = torch.ones(d, dtype=torch.bool, device=theta.device)
        mask[k] = False
        selected = theta[:, mask]
        coordinate = self.coord_embedding[mask].view(1, d - 1, self.coord_embed_dim).expand(batch, d - 1, self.coord_embed_dim)
        boundary = self.boundary_embedding[k].view(1, 1, self.boundary_embed_dim).expand(batch, d - 1, self.boundary_embed_dim)
        return torch.cat([selected.unsqueeze(-1), coordinate, boundary], dim=2).reshape(batch * (d - 1), 1 + self.coord_embed_dim + self.boundary_embed_dim), mask

    def _boundary_zero_inputs_for_k(self, theta, k):
        mask = torch.ones(self.d, dtype=torch.bool, device=theta.device)
        mask[k] = False
        zeros = torch.zeros(self.d - 1, 1, dtype=c_dtype, device=theta.device)
        coordinate = self.coord_embedding[mask]
        boundary = self.boundary_embedding[k].view(1, self.boundary_embed_dim).expand(self.d - 1, self.boundary_embed_dim)
        return torch.cat([zeros, coordinate, boundary], dim=1)

    def _boundary_fk_and_deriv_for_k(self, theta, k, need_derivative):
        batch, d = theta.shape
        inputs, mask = self._boundary_inputs_for_k(theta, k)
        zero_inputs = self._boundary_zero_inputs_for_k(theta, k)
        if need_derivative:
            values, derivatives = run_scalar_net_chunked(self.boundary_net, inputs, self.pair_chunk_size, derivative_mode="first")
            derivatives = derivatives.view(batch, d - 1)
        else:
            values = run_scalar_net_chunked(self.boundary_net, inputs, self.pair_chunk_size)
            derivatives = None
        zero_values = run_scalar_net_chunked(self.boundary_net, zero_inputs, self.pair_chunk_size).view(1, d - 1)
        values = values.view(batch, d - 1)
        fk = self.boundary_log_mass[k].view(1, 1) + torch.sum(values - zero_values, dim=1, keepdim=True)
        return fk, derivatives, mask

    def forward_selected_boundary(self, theta, k):
        theta = theta.to(dtype=c_dtype, device=device)
        f0, _ = self._interior_f0_and_jac(theta, need_jacobian=False)
        fk, _, _ = self._boundary_fk_and_deriv_for_k(theta, k, need_derivative=False)
        return f0, fk

    def forward(self, theta):
        values, _ = self.forward_with_input_jacobian(theta, need_jacobian=False)
        return values

    def forward_with_input_jacobian(self, theta, need_jacobian=True):
        theta = theta.to(dtype=c_dtype, device=device)
        if theta.ndim != 2 or theta.shape[1] != self.d:
            raise ValueError(f"theta must have shape [B,{self.d}], got {tuple(theta.shape)}")
        batch, d = theta.shape
        f0, joint_jacobian = self._interior_f0_and_jac(theta, need_jacobian)
        boundary_columns = []
        jacobian = torch.zeros(batch, d + 1, d, dtype=c_dtype, device=theta.device) if need_jacobian else None
        if need_jacobian:
            jacobian[:, 0, :] = joint_jacobian
        for k in range(d):
            fk, derivatives, mask = self._boundary_fk_and_deriv_for_k(theta, k, need_jacobian)
            boundary_columns.append(fk)
            if need_jacobian:
                jacobian[:, k + 1, mask] = derivatives
        return torch.cat([f0] + boundary_columns, dim=1), jacobian



## 边界交互扩展
共享边界global网络读取九个剩余坐标和边界embedding；删除法向坐标，解析Jacobian对应位置严格为零。两组消融的参数量会分别记录，默认不声称参数量匹配。

In [ ]:
class D10HCN(StructuredNonproductGlobalCauchyMGF):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.interactions = VARIANT == 'interaction'
        if self.interactions:
            self.boundary_global_net = HolomorphicCauchyScalarNet(
                self.d-1+self.boundary_embed_dim, MODEL_CONFIG['global_hidden_width'])
        else:
            del self.global_net

    def _interior_f0_and_jac(self, theta, need_jacobian):
        if self.interactions:
            return super()._interior_f0_and_jac(theta, need_jacobian)
        b,d=theta.shape
        x=self._interior_inputs(theta)
        if need_jacobian:
            v,j=run_scalar_net_chunked(self.interior_net,x,self.pair_chunk_size,'first')
            j=j.reshape(b,d)
        else:
            v=run_scalar_net_chunked(self.interior_net,x,self.pair_chunk_size); j=None
        z=torch.cat([torch.zeros(d,1,dtype=c_dtype,device=device),self.coord_embedding],1)
        v0=self.interior_net(z).reshape(1,d)
        return (v.reshape(b,d)-v0).sum(1,keepdim=True),j

    def _boundary_fk_and_deriv_for_k(self, theta,k,need_derivative):
        f,j,mask=super()._boundary_fk_and_deriv_for_k(theta,k,need_derivative)
        if not self.interactions: return f,j,mask
        b=theta.shape[0]
        emb=self.boundary_embedding[k:k+1]
        x=torch.cat([theta[:,mask],emb.expand(b,-1)],1)
        z=torch.cat([torch.zeros(1,self.d-1,dtype=c_dtype,device=device),emb],1)
        if need_derivative:
            g,dg=run_scalar_net_chunked(self.boundary_global_net,x,self.pair_chunk_size,'all')
            j=j+dg[:,:self.d-1]
        else:
            g=run_scalar_net_chunked(self.boundary_global_net,x,self.pair_chunk_size)
        return f+g-self.boundary_global_net(z),j,mask

def build_model():
    set_all_seeds(SEED)
    return D10HCN(**MODEL_CONFIG,boundary_mass=compute_boundary_mass(PARAMS)).to(device)

def count_params(model):
    return sum(p.numel()*(2 if p.is_complex() else 1) for p in model.parameters())


In [ ]:
def sample_complex_box(N, d=2, re_min=-0.5, re_max=5.0, im_max=5.0, requires_grad=False, device=device):
    real = re_min + (re_max - re_min) * torch.rand(N, d, dtype=r_dtype, device=device)
    imag = -im_max + 2.0 * im_max * torch.rand(N, d, dtype=r_dtype, device=device)
    theta = torch.complex(real, imag)
    if requires_grad:
        theta.requires_grad_(True)
    return theta


def sample_nn4mgf_vector_mgf(N, d=2, lb=-5.0, ub=0.5, imag_lb=-5.0, imag_ub=5.0, device=device):
    real_lb_curr = ub - (ub - lb) * torch.rand(N, 1, dtype=r_dtype, device=device)
    imag_bd = max(abs(imag_ub), abs(imag_lb)) * torch.rand(N, 1, dtype=r_dtype, device=device)
    real = (ub - real_lb_curr) * torch.rand(N, d, dtype=r_dtype, device=device) + real_lb_curr
    imag = 2.0 * imag_bd * torch.rand(N, d, dtype=r_dtype, device=device) - imag_bd
    return torch.complex(real, imag)


def sample_nn4mgf_vector_laplace(N, d=2, lap_re_min=-0.5, lap_re_max=5.0, im_max=5.0, requires_grad=False, device=device):
    theta = -sample_nn4mgf_vector_mgf(N=N, d=d, lb=-lap_re_max, ub=-lap_re_min, imag_lb=-im_max, imag_ub=im_max, device=device)
    if requires_grad:
        theta.requires_grad_(True)
    return theta


def sample_training_theta(N, d=2, re_min=-0.5, re_max=5.0, im_max=5.0, requires_grad=False, device=device):
    return sample_nn4mgf_vector_laplace(N=N, d=d, lap_re_min=re_min, lap_re_max=re_max, im_max=im_max, requires_grad=requires_grad, device=device)


def sample_nested_real_radius(N, lb=0.0, ub=5.0, device=device):
    # Near-zero-biased sampler: E[s] = lb + (ub-lb)/3 for U(0,1)^2.
    u = torch.rand(N, 1, dtype=r_dtype, device=device)
    return lb + (ub - lb) * (u ** 2)


def sample_mixed_bar_theta(N=4000, d=2, re_min=-0.5, re_max=5.0, im_max=5.0, device=device):
    """
    Mixed 4000-point BAR sampler proportions:
      2000 global, 800 exact complex diagonal, 800 near-diagonal complex, 400 positive-real diagonal.
    Counts are scaled proportionally for N != 4000.
    """
    n_global = int(round(N * 0.50))
    n_exact_diag = int(round(N * 0.20))
    n_near_diag = int(round(N * 0.20))
    n_pos_diag = N - n_global - n_exact_diag - n_near_diag

    global_theta = sample_training_theta(n_global, d=d, re_min=re_min, re_max=re_max, im_max=im_max, device=device)
    s_complex = sample_training_theta(n_exact_diag, d=1, re_min=re_min, re_max=re_max, im_max=im_max, device=device)
    exact_diag = s_complex.expand(n_exact_diag, d)

    s_near = sample_training_theta(n_near_diag, d=1, re_min=re_min, re_max=re_max, im_max=im_max, device=device)
    noise = 0.05 * sample_complex_box(n_near_diag, d=d, re_min=-1.0, re_max=1.0, im_max=1.0, device=device)
    near_diag_raw = s_near.expand(n_near_diag, d) + noise
    near_diag = torch.complex(
        torch.clamp(near_diag_raw.real, min=re_min, max=re_max),
        torch.clamp(near_diag_raw.imag, min=-im_max, max=im_max),
    )

    positive_lb = max(0.0, re_min)
    positive_ub = re_max
    if positive_ub <= positive_lb:
        raise ValueError(f"Positive-real diagonal domain is empty: [{positive_lb}, {positive_ub}]")
    s_real = sample_nested_real_radius(n_pos_diag, lb=positive_lb, ub=positive_ub, device=device)
    pos_diag = torch.complex(s_real.expand(n_pos_diag, d), torch.zeros(n_pos_diag, d, dtype=r_dtype, device=device))

    theta = torch.cat([global_theta, exact_diag, near_diag, pos_diag], dim=0).to(dtype=c_dtype, device=device)
    return theta[torch.randperm(theta.shape[0], device=device)]

In [ ]:
def construct_boundary_directions(params):
    R_c = params.R.to(dtype=c_dtype, device=device)
    A = R_c.T
    d = R_c.shape[0]
    eye = torch.eye(d, dtype=c_dtype, device=device)
    rows = []
    for k in range(d):
        M = A.clone()
        M[k, :] = eye[k, :]
        rows.append(torch.linalg.solve(M, eye[k, :]).view(1, d))
    return torch.cat(rows, dim=0)


def construct_boundary_thetas(theta, params):
    theta = theta.to(dtype=c_dtype, device=device)
    _, d = theta.shape
    V = construct_boundary_directions(params)
    return [theta[:, k:k+1] * V[k:k+1, :] for k in range(d)], V


# ============================================================
# 8-10. BAR, structured boundary loss, fixed held-out evaluation
# ============================================================

DEFAULT_ADAPTIVE_BAR_CONFIG = {
    "adaptive_bar_weight": True,
    "bar_weight_alpha": 0.3,
    "bar_weight_eps": 1e-12,
    "bar_weight_min": 0.5,
    "bar_weight_max": 2.0,
}


def with_adaptive_bar_defaults(config):
    merged = dict(DEFAULT_ADAPTIVE_BAR_CONFIG)
    merged.update(config)
    return merged


def adaptive_bar_kwargs(config):
    cfg = with_adaptive_bar_defaults(config)
    return {
        "adaptive_bar_weight": bool(cfg["adaptive_bar_weight"]),
        "bar_weight_alpha": float(cfg["bar_weight_alpha"]),
        "bar_weight_eps": float(cfg["bar_weight_eps"]),
        "bar_weight_min": float(cfg["bar_weight_min"]),
        "bar_weight_max": float(cfg["bar_weight_max"]),
    }


def split_f_tuple(f_out):
    if isinstance(f_out, torch.Tensor):
        return f_out
    if isinstance(f_out, (tuple, list)):
        return torch.cat(list(f_out), dim=1)
    raise TypeError(f"Unsupported model output type {type(f_out)!r}")


def compute_bar_pointwise(theta, f_tuple, params, eps_gamma=1e-12, eps_bar=1e-8):
    f_mat = split_f_tuple(f_tuple)
    gamma0, gamma_all = compute_gammas(theta, params)
    gamma_mat = torch.cat([gamma0, gamma_all], dim=1)
    nu = torch.max(torch.log(torch.abs(gamma_mat) + eps_gamma) + f_mat.real, dim=1, keepdim=True)[0]
    kappa = gamma_mat * torch.exp(f_mat - nu.to(dtype=c_dtype))
    left = kappa[:, 0:1]
    right = torch.sum(kappa[:, 1:], dim=1, keepdim=True)
    return (torch.abs(left - right) / (torch.abs(left) + torch.abs(right) + eps_bar)) ** 2


def weighted_normalized_mean(pointwise, adaptive_bar_weight=True, bar_weight_alpha=0.3, bar_weight_eps=1e-12, bar_weight_min=0.5, bar_weight_max=2.0):
    unweighted = torch.mean(pointwise)
    if adaptive_bar_weight:
        raw_w = (pointwise.detach() + bar_weight_eps) ** bar_weight_alpha
        weights = torch.clamp(raw_w / (torch.mean(raw_w) + bar_weight_eps), min=bar_weight_min, max=bar_weight_max).detach()
        weighted = torch.sum(weights * pointwise) / (torch.sum(weights) + bar_weight_eps)
    else:
        weights = torch.ones_like(pointwise).detach()
        weighted = unweighted
    return weighted, unweighted, weights


def compute_bar_loss(theta, f_tuple, params, eps_gamma=1e-12, eps_bar=1e-8, adaptive_bar_weight=True, bar_weight_alpha=0.3, bar_weight_eps=1e-12, bar_weight_min=0.5, bar_weight_max=2.0, return_diagnostics=False):
    pointwise = compute_bar_pointwise(theta, f_tuple, params, eps_gamma=eps_gamma, eps_bar=eps_bar)
    weighted, unweighted, weights = weighted_normalized_mean(
        pointwise,
        adaptive_bar_weight=adaptive_bar_weight,
        bar_weight_alpha=bar_weight_alpha,
        bar_weight_eps=bar_weight_eps,
        bar_weight_min=bar_weight_min,
        bar_weight_max=bar_weight_max,
    )
    if not return_diagnostics:
        return weighted
    return weighted, {
        "bar_unweighted": unweighted.detach(),
        "bar_weighted": weighted.detach(),
        "bar_weight_mean": torch.mean(weights).detach(),
        "bar_weight_min": torch.min(weights).detach(),
        "bar_weight_max": torch.max(weights).detach(),
    }


def boundary_face_residual_from_f(theta_k, f0, fk, params, k, eps_gamma=1e-12, eps_bar=1e-8):
    gamma0, gamma_all = compute_gammas(theta_k, params)
    gammak = gamma_all[:, k:k+1]
    nu = torch.maximum(torch.log(torch.abs(gamma0) + eps_gamma) + f0.real, torch.log(torch.abs(gammak) + eps_gamma) + fk.real).to(dtype=c_dtype)
    left = gamma0 * torch.exp(f0 - nu)
    right = gammak * torch.exp(fk - nu)
    rel = torch.abs(left - right) / (torch.abs(left) + torch.abs(right) + eps_bar)
    return rel ** 2


def compute_boundary_multiplicative_loss_generic(model, theta, params, eps_gamma=1e-12, eps_bar=1e-8):
    theta = theta.to(dtype=c_dtype, device=device)
    _, d = theta.shape
    tilde_thetas, _ = construct_boundary_thetas(theta, params)
    total = torch.zeros((), dtype=r_dtype, device=device)
    for k, theta_k in enumerate(tilde_thetas):
        f_mat = split_f_tuple(model(theta_k))
        total = total + torch.mean(boundary_face_residual_from_f(theta_k, f_mat[:, 0:1], f_mat[:, k+1:k+2], params, k, eps_gamma, eps_bar))
    return total / d


def compute_boundary_multiplicative_loss_structured(model, theta, params, eps_gamma=1e-12, eps_bar=1e-8):
    theta = theta.to(dtype=c_dtype, device=device)
    _, d = theta.shape
    tilde_thetas, _ = construct_boundary_thetas(theta, params)
    total = torch.zeros((), dtype=r_dtype, device=device)
    for k, theta_k in enumerate(tilde_thetas):
        f0, fk = model.forward_selected_boundary(theta_k, k)
        total = total + torch.mean(boundary_face_residual_from_f(theta_k, f0, fk, params, k, eps_gamma, eps_bar))
    return total / d


def evaluate_boundary_residual_independent(model, theta_bdary_eval, params, eps_gamma=1e-12, eps_bar=1e-8):
    theta = theta_bdary_eval.to(dtype=c_dtype, device=device)
    _, d = theta.shape
    tilde_thetas, _ = construct_boundary_thetas(theta, params)
    face_means = []
    for k, theta_k in enumerate(tilde_thetas):
        f0, fk = model.forward_selected_boundary(theta_k, k)
        face_means.append(torch.mean(boundary_face_residual_from_f(theta_k, f0, fk, params, k, eps_gamma, eps_bar)))
    face_tensor = torch.stack(face_means)
    return {
        "eval_boundary_mean": torch.mean(face_tensor),
        "eval_boundary_max": torch.max(face_tensor),
        "eval_boundary_worst_face": int(torch.argmax(face_tensor).detach().cpu()),
        "eval_boundary_per_face": face_tensor.detach(),
    }


def compute_zero_loss(model, params):
    theta0 = torch.zeros(1, params.R.shape[0], dtype=c_dtype, device=device)
    return torch.mean(torch.abs(torch.exp(model(theta0)[:, 0:1]) - 1.0) ** 2)


def compute_monotonicity_components_analytical_from_theta_real(model, theta_real):
    theta = torch.complex(theta_real.to(dtype=r_dtype, device=device), torch.zeros_like(theta_real.to(dtype=r_dtype, device=device)))
    f_mat, J = model.forward_with_input_jacobian(theta, need_jacobian=True)
    loss_mono = torch.mean(torch.relu(J.real) ** 2)
    loss_imag = torch.mean(f_mat.imag ** 2)
    return loss_mono, loss_imag


def compute_monotonicity_components_analytical(model, N_mono, d=2, re_min=-0.5, re_max=5.0):
    theta = sample_training_theta(N_mono, d=d, re_min=re_min, re_max=re_max, im_max=0.0, device=device)
    return compute_monotonicity_components_analytical_from_theta_real(model, theta.real.detach())




In [ ]:
def training_step(model,optimizer,batches):
    model.train(); optimizer.zero_grad(set_to_none=True)
    reset_all_cauchy_denominator_tracking(model)
    nb,nd,nm=batches
    # Sample first so both variants consume the same sampling stream.
    bar=sample_mixed_bar_theta(nb,d=D,re_min=RE_MIN,re_max=RE_MAX,im_max=IM_MAX)
    bd=sample_training_theta(nd,d=D,re_min=RE_MIN,re_max=RE_MAX,im_max=IM_MAX)
    mono=sample_training_theta(nm,d=D,re_min=0.,re_max=RE_MAX,im_max=0.).real
    vals={}
    def backward(loss,name):
        if not torch.isfinite(loss): raise FloatingPointError(name+' is nonfinite')
        vals[name]=float(loss.detach().cpu()); loss.backward()
    backward(compute_bar_loss(bar,model(bar),PARAMS), 'bar')
    # Each face is backpropagated separately, with the original 1/d weight.
    bd_value=0.
    for k in range(D):
        theta=torch.zeros_like(bd); theta[:,k]=bd[:,k]  # R=I
        f0,fk=model.forward_selected_boundary(theta,k)
        loss=boundary_face_residual_from_f(theta,f0,fk,PARAMS,k).mean()/D
        backward(loss,'boundary_face'); bd_value+=vals['boundary_face']
    vals.pop('boundary_face'); vals['boundary']=bd_value
    lm,li=compute_monotonicity_components_analytical_from_theta_real(model,mono)
    vals['mono']=float(lm.detach().cpu()); vals['imag']=float(li.detach().cpu())
    backward(lm+li,'real_constraints');vals.pop('real_constraints')
    for name,p in model.named_parameters():
        if p.grad is not None and not torch.isfinite(p.grad).all():
            raise FloatingPointError('Nonfinite gradient: '+name)
    distance=minimum_tracked_cauchy_denominator(model)
    if not math.isfinite(distance) or distance<=POLE_TOL:
        raise FloatingPointError('Pole proximity: '+str(distance))
    grad=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
    optimizer.step()
    vals['total']=sum(vals.values());vals['grad_norm']=float(grad)
    vals['min_denominator']=distance
    return vals

def preflight():
    m=build_model(); z=torch.zeros(2,D,dtype=c_dtype,device=device)
    f,J=m.forward_with_input_jacobian(z)
    assert torch.max(torch.abs(f))<1e-10
    for k in range(D): assert torch.max(torch.abs(J[:,k+1,k]))==0
    x=torch.complex(torch.rand(2,D,device=device)*0.15,torch.rand(2,D,device=device)*0.05)
    _,J=m.forward_with_input_jacobian(x); eps=1e-6; errors=[]
    for k in range(D):
        dx=torch.zeros_like(x);dx[:,k]=eps
        fd=(m(x+dx)-m(x-dx))/(2*eps)
        errors.append(float((fd-J[:,:,k]).abs().max().detach()))
    assert max(errors)<1e-5,errors
    opt=torch.optim.Adam(m.parameters(),lr=LR_INITIAL)
    row=training_step(m,opt,(8,2,2))
    groups={}
    for name,p in m.named_parameters():
        groups.setdefault(name.split('.')[0],0.)
        if p.grad is not None:groups[name.split('.')[0]]+=float(p.grad.abs().sum().detach())
    assert all(v>0 for v in groups.values()),groups
    report=dict(real_parameters=count_params(m),jacobian_max_error=max(errors),
                gradient_groups=groups,smoke_loss=row)
    print('Preflight passed:',report)
    return report

PREFLIGHT=preflight()


In [ ]:
def rng_state():
    return dict(python=random.getstate(),numpy=np.random.get_state(),torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)

def restore_rng(r):
    random.setstate(r['python']);np.random.set_state(r['numpy']);torch.set_rng_state(r['torch'].cpu())
    if r['cuda'] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([x.cpu() for x in r['cuda']])

def save_checkpoint(path,model,opt,step,elapsed,history):
    tmp=path.with_suffix('.tmp')
    torch.save(dict(config=CONFIG,model=model.state_dict(),optimizer=opt.state_dict(),
                    step=step,elapsed=elapsed,history=history,rng=rng_state()),tmp)
    tmp.replace(path)

def run_training():
    resume=RUN_MODE=='resume'
    if resume:
        if RUN_DIR is None: raise ValueError('Set RUN_DIR to the original output folder')
        folder=Path(RUN_DIR); payload=safe_torch_load(folder/'checkpoint_latest.pt')
        if payload['config']!=CONFIG: raise ValueError('Resume configuration does not match')
    else:
        folder=Path('outputs')/('d10_'+VARIANT+'_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        folder.mkdir(parents=True,exist_ok=False)
    print('RUN DIRECTORY:',folder.resolve())
    model=build_model(); opt=torch.optim.Adam(model.parameters(),lr=LR_INITIAL)
    start,elapsed,history=0,0.,[]
    if resume:
        model.load_state_dict(payload['model']);opt.load_state_dict(payload['optimizer'])
        start,elapsed,history=payload['step'],payload['elapsed'],payload['history']
        restore_rng(payload['rng'])
    else:
        set_all_seeds(SEED+1)  # training sampler independent of initialization size
    write_json(folder/'config.json',CONFIG)
    write_json(folder/'preflight.json',PREFLIGHT)
    write_json(folder/'environment.json',dict(torch=torch.__version__,device=str(device),
        gpu=torch.cuda.get_device_name(0) if device.type=='cuda' else None))
    if device.type=='cuda':torch.cuda.reset_peak_memory_stats()
    cuda_sync(); clock=time.perf_counter()
    for step in range(start+1,TRAIN_STEPS+1):
        for group in opt.param_groups:group['lr']=LR_INITIAL if step<=LR_SWITCH else LR_FINAL
        row=training_step(model,opt,(BATCH_BAR,BATCH_BOUNDARY,BATCH_MONO))
        if step%LOG_EVERY==0 or step==TRAIN_STEPS:
            cuda_sync(); row.update(step=step,seconds=elapsed+time.perf_counter()-clock,
                lr=opt.param_groups[0]['lr'])
            history.append(row);write_csv(folder/'training_log.csv',history,list(row))
            print(f"step {step}/{TRAIN_STEPS} total={row['total']:.4g} elapsed={row['seconds']/3600:.2f}h",flush=True)
        if step%SAVE_EVERY==0 or step==TRAIN_STEPS:
            cuda_sync()
            save_checkpoint(folder/'checkpoint_latest.pt',model,opt,step,
                            elapsed+time.perf_counter()-clock,history)
    save_checkpoint(folder/f'checkpoint_step{TRAIN_STEPS}.pt',model,opt,TRAIN_STEPS,
                    elapsed+time.perf_counter()-clock,history)
    write_json(folder/'resource_summary.json',dict(real_parameters=count_params(model),
        base_inputs=TRAIN_STEPS*sum(CONFIG['batch']),
        peak_allocated_bytes=torch.cuda.max_memory_allocated() if device.type=='cuda' else 0,
        memory_scope='current session',training_seconds=elapsed+time.perf_counter()-clock))
    return folder

FINISHED_DIR=None
if RUN_MODE in {'train','resume'}:FINISHED_DIR=run_training()
elif RUN_MODE=='evaluate':
    if RUN_DIR is None:raise ValueError('Set RUN_DIR')
    FINISHED_DIR=Path(RUN_DIR)
else:print('Smoke passed. Set RUN_MODE="train", restart kernel and Run All.')


## 训练后评估
下面的模拟参考来自用户提供的截图，尚未核验原始模拟与离散偏差，不参与训练。
可以设置 REFERENCE_CSV 为含 t、reference 列的CSV（可附 standard_error、ci_low、ci_high）。请仅加载同一10D/rho=0.3系统的参考。
Degree 8为pilot主指标，Degree 5为诊断；degree提高不保证本系统误差必然降低。两者都在同一最终权重上计算。mpmath的工作精度不改变网络complex128精度。

In [ ]:
REFERENCE_CSV = None
T_VALUES=list(range(1,9))
PILOT=[0.97204,0.72091,0.40030,0.19124,0.08443,0.03576,0.014645,0.005905]

def evaluate(folder):
    path=folder/f'checkpoint_step{TRAIN_STEPS}.pt'
    before=sha256_file(path);p=safe_torch_load(path)
    if p['config']!=CONFIG or p['step']!=TRAIN_STEPS:raise ValueError('Endpoint/config mismatch')
    m=build_model();m.load_state_dict(p['model']);m.eval()
    ref={t:dict(reference=v) for t,v in zip(T_VALUES,PILOT)}
    status='unverified screenshot pilot; no certified discretization error'
    if REFERENCE_CSV is not None:
        with open(REFERENCE_CSV,newline='') as f:
            ref={int(r['t']):r for r in csv.DictReader(f)}
        status='external numerical reference: '+str(REFERENCE_CSV)
    rows=[]
    with torch.no_grad():
        _,j=m.forward_with_input_jacobian(torch.zeros(1,D,dtype=c_dtype,device=device))
        mean=float(-j[0,0].sum().real)
    for degree in [5,8]:
        for t in T_VALUES:
            def tail_transform(s):
                if abs(s)<mp.mpf('1e-30'):return mp.mpf(mean)
                theta=torch.full((1,D),complex(s),dtype=c_dtype,device=device)
                with torch.no_grad():v=torch.exp(m._interior_f0_and_jac(theta,False)[0][0,0]).item()
                return (1-mp.mpc(v.real,v.imag))/s
            error=''
            try:
                with mp.workdps(50):pred=float(mp.invertlaplace(tail_transform,t,method='talbot',degree=degree))
            except Exception as exc:pred=float('nan');error=str(exc)
            truth=float(ref[t]['reference'])
            if not 0<truth<=1:raise ValueError('Invalid reference')
            rows.append(dict(degree=degree,t=t,prediction=pred,reference=truth,
                relative_error_percent=100*abs(pred-truth)/truth,
                independent_tail=-math.expm1(D*math.log1p(-math.exp(-t))),
                invalid=not math.isfinite(pred) or not 0<=pred<=1,error=error))
    write_csv(folder/'pointwise.csv',rows,list(rows[0]))
    write_json(folder/'reference_used.json',dict(status=status,values=ref))
    summary={}
    for degree in [5,8]:
        rr=[r for r in rows if r['degree']==degree]
        summary[str(degree)]={f'mre_1_{end}_percent':float(np.mean([r['relative_error_percent'] for r in rr if r['t']<=end])) for end in [5,8]}
        summary[str(degree)]['invalid_count']=sum(r['invalid'] for r in rr)
    summary['reference_status']=status;summary['learned_sum_mean']=mean
    summary['known_sum_mean']=sum(1/i for i in range(1,D+1))
    summary['checkpoint_sha256']=before
    assert before==sha256_file(path)
    write_json(folder/'summary.json',summary)
    fig,ax=plt.subplots(figsize=(7,4.5))
    rr=[r for r in rows if r['degree']==8]
    ax.semilogy(T_VALUES,[r['reference'] for r in rr],'ko-',label='Numerical reference (pilot)')
    ax.semilogy(T_VALUES,[r['independent_tail'] for r in rr],'--',label='Independent marginals')
    ax.semilogy(T_VALUES,[r['prediction'] if r['prediction']>0 else np.nan for r in rr],'s-',label=VARIANT+', Degree 8')
    ax.set(xlabel='t',ylabel='P(sum Z > t)');ax.legend();ax.grid(alpha=.25);fig.tight_layout()
    fig.savefig(folder/'tail_comparison.png',dpi=180);plt.show()
    print(json.dumps(summary,indent=2))
    return summary

if FINISHED_DIR is not None:SUMMARY=evaluate(FINISHED_DIR)


## 本版本已完成的检查
CPU / float64-complex128，小批量：两种结构均通过原点锚定、边界法向导数为零、全部输入维度解析Jacobian有限差分对照及模块梯度检查。interaction真实标量参数量23,925；coordinate-only为14,329，两者不是等参数消融。保存恢复后下一训练步的参数逐位一致。两步训练到Degree 5/8评估与文件导出流程已跑通。

上述检查不是完整训练精度或GPU显存验证。pilot参考不会进入训练。每1000步保存最近检查点，意外中断最多需要重跑最近1000步；恢复配置必须与原运行一致。负预测等无效输出保留在CSV；对数图不显示非正值。